## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
from PIL import Image
import csv
import shutil
import configparser
from collections import Counter
import numpy as np

from qdrant_client import QdrantClient, models

In [ ]:
from sem_rag.config.settings import Qdrant, LAMEDataset

## Import Images

In [ ]:
image_extensions = [".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"]

image_paths = sorted([path for path in LAMEDataset.SOURCE_DIR.rglob("*") if path.is_file() and path.suffix.lower() in image_extensions])
count = 0
for image in image_paths:
    print(image)
    count += 1

print(count)

In [ ]:
image = image_paths[0]
image

In [ ]:
# img = Image.open(LAMEDataset.SOURCE_DIR / "FIB maintainence/spots 2026-06-22 and 30/30keV 100pA.png")
img = Image.open(image)

In [ ]:
print(img.size, img.mode)

In [ ]:
arr = np.array(img)
arr.ndim

In [ ]:
# img = img.convert("L")
img

In [ ]:
def cropping(img):
    w, h = img.size
    res = img.crop((w//2 - 256, h//2 - 256, w//2 + 256, h//2 + 256))
    return res

In [ ]:
res = cropping(img)
res

In [ ]:
res.size

In [ ]:
img_array = np.array(res)

In [ ]:
img_array

## Image Processing

For each image, we check whether it has a corresponding .hdr file, which contains the metadata, and if it is bigger than $512\times512$.

In [ ]:
# Add or remove extensions depending on your dataset
image_extensions = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

min_size = 512

valid_count = 0
missing_hdr_count = 0
small_image_count = 0
unreadable_count = 0

# Collect files before copying anything
images = [path for path in LAMEDataset.SOURCE_DIR.rglob("*") if path.is_file() and path.suffix.lower() in image_extensions]

for image_path in images:
    # Find the matching .hdr file, ignoring extension case
    hdr_path = next(
        (
            sibling
            for sibling in image_path.parent.iterdir()
            if sibling.is_file()
            and sibling.stem == image_path.stem + "-" + image_path.suffix.strip(".")
            and sibling.suffix.lower() == ".hdr"
        ),
        None,
    )

    # Criterion 1: matching metadata file exists
    if hdr_path is None:
        print(f"MISSING HDR: {image_path}")
        missing_hdr_count += 1
        continue

    # Criterion 2: both dimensions are >= 512
    try:
        with Image.open(image_path) as img:
            width, height = img.size

        if width < min_size or height < min_size:
            print(f"TOO SMALL: {image_path} ({width}x{height})")
            small_image_count += 1
            continue

    except Exception as exc:
        print(f"UNREADABLE: {image_path} ({exc})")
        unreadable_count += 1
        continue

    valid_count += 1

In [ ]:
print("\n--- Summary ---")
print(f"Images scanned:       {len(images)}")
print(f"Valid images:         {valid_count}")
print(f"Missing HDR:          {missing_hdr_count}")
print(f"Images too small:     {small_image_count}")
print(f"Unreadable images:    {unreadable_count}")

## Image Saving

In [ ]:
LAMEDataset.LAME_DIR.mkdir(parents=True, exist_ok=True)
LAMEDataset.CROPS.mkdir(parents=True, exist_ok=True)
LAMEDataset.META.mkdir(parents=True, exist_ok=True)

In [ ]:
image_extensions = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}
min_size = 512

In [ ]:
def cropping(img):
    w, h = img.size
    res = img.crop((w//2 - 256, h//2 - 256, w//2 + 256, h//2 + 256))
    return res

In [ ]:
images = [path for path in LAMEDataset.SOURCE_DIR.rglob("*") if path.is_file() and path.suffix.lower() in image_extensions]
print(len(images))

manifest = []

In [ ]:
"__".join(images[3000].relative_to(LAMEDataset.SOURCE_DIR).with_suffix("").parts).replace(" ", "-")

In [ ]:
def to_grayscale(img: Image.Image) -> Image.Image:
    array = np.array(img)

    if array.ndim == 2:
        if array.dtype == np.uint16:
            array = (array / 257).round().astype(np.uint8)
            return Image.fromarray(array)

    return img.convert("L")

It is very similar to the loop above, but this time we are also cropping the image, converting it to grayscale, and saving it into a new folder, alongside its associated .hdr file.

In [ ]:
for image_path in images:
    # Find the matching .hdr file, ignoring extension case
    hdr_path = next((
            sibling
            for sibling in image_path.parent.iterdir()
            if sibling.is_file()
            and sibling.stem == image_path.stem + "-" + image_path.suffix.strip(".")
            and sibling.suffix.lower() == ".hdr"
        ), None,
    )

    if hdr_path is None:
        continue

    try:
        with Image.open(image_path) as img:
            width, height = img.size
            if width < min_size or height < min_size:
                continue

            cropped = cropping(img)
            cropped = to_grayscale(cropped)
            array = np.array(cropped)

    except Exception as exc:
        continue

    relative_path = image_path.relative_to(LAMEDataset.SOURCE_DIR)
    output_name = "__".join(relative_path.with_suffix("").parts).replace(" ", "-")

    # Save the array and copy the corresponding header
    array_path = LAMEDataset.CROPS / f"{output_name}.npy"
    output_hdr_path = LAMEDataset.META / f"{output_name}.hdr"

    np.save(array_path, array)
    shutil.copy2(hdr_path, output_hdr_path)

    manifest.append({
        "array_file": array_path.name,
        "header_file": output_hdr_path.name,
        "original_image": str(relative_path),
        "original_header": str(hdr_path.relative_to(LAMEDataset.SOURCE_DIR)),
    })

Save a .csv manifest to map the saved images to the original path.

In [ ]:
# Write the manifest
manifest_path = LAMEDataset.LAME_DIR / "manifest.csv"

with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=manifest[0].keys() if manifest else [
        "array_file", "header_file", "original_image",
        "original_header", "width", "height",
        "array_shape", "array_dtype",
    ])
    writer.writeheader()
    writer.writerows(manifest)

print(f"\nProcessed {len(manifest)} images.")
print(f"Dataset saved to: {LAMEDataset.LAME_DIR}")

In [ ]:
images = [path for path in LAMEDataset.CROPS.rglob("*")]
len(images)

In [ ]:
print("Manifest entries:", len(manifest))
print("Unique array filenames:", len({
    item["array_file"] for item in manifest
}))
print("Unique header filenames:", len({
    item["header_file"] for item in manifest
}))

In [ ]:
emb = [path for path in LAMEDataset.EMBEDDINGS_QWEN.rglob("*")]
len(emb)

In [ ]:
FIELDS = {
    "scan_mode": ("SEM", "ScanMode"),
    "detector": ("SEM", "Detector"),
}

values = {field: Counter() for field in FIELDS}

for hdr_path in LAMEDataset.META.rglob("*.hdr"):
    parser = configparser.ConfigParser(interpolation=None)
    parser.optionxform = str

    try:
        with hdr_path.open("r", encoding="utf-8-sig") as f:
            parser.read_file(f)
    except (OSError, configparser.Error) as e:
        print(f"Could not read {hdr_path}: {e}")
        continue

    for field, (section, key) in FIELDS.items():
        value = parser.get(section, key, fallback=None)

        if value and value.strip():
            values[field][value.strip()] += 1

for field, counts in values.items():
    print(f"\n{field} ({len(counts)} unique values)")
    for value, count in sorted(counts.items(), key=lambda item: item[0].casefold()):
        print(f"  {value!r}: {count}")

## Qdrant Retrieval Testing

In [ ]:
client = QdrantClient(url=Qdrant.URL, port=80)

In [ ]:
result = client.scroll(
    collection_name=Qdrant.COLLECTION_LAME,
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="meta.detector",
                match=models.MatchValue(value="SE"),
            )
        ]
    ),
    limit=5,
    with_payload=True,
    with_vectors=False,
)

for point in result[0]:
    print(point.payload)

In [ ]:
points, _ = client.scroll(
    collection_name=Qdrant.COLLECTION_LAME,
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="meta.image_id",
                match=models.MatchValue(value="FIB-maintainence__spots-2026-09-21__30keV_20pA_2"),
            )
        ]
    ),
    with_payload=["meta.thumb_b64"],
    limit=1,
)

In [ ]:
points